In [1]:
import sys

import pm4py

import pandas as pd
import numpy as np

import torch
from torch.utils.data import DataLoader

from sklearn.model_selection import train_test_split

from utils.general_utils import set_stdout_to_file, set_seed

from config.feature_config import FeatureConfig

from model.preprocessor import PreprocessorArtifacts
from model.next_event_model import ProcessLSTM, train_ProcessLSTM, validate_ProcessLSTM

### --- Preprocess dataset ---

In [2]:
set_seed(seed=42)

In [3]:
log = pm4py.read_xes("../../data/bpic17.xes")

C:\Users\dcoralage\Downloads\counterfactual_exp\counterfactual_env\lib\site-packages\pm4py\utils.py:1027: UserWarning: Install the optional requirement `r4pm` to import/export files faster. `rustxes` remains supported as a fallback.
  warnings.warn(
C:\Users\dcoralage\Downloads\counterfactual_exp\counterfactual_env\lib\site-packages\pm4py\util\dt_parsing\parser.py:82: UserWarning: ISO8601 strings are not fully supported with strpfromiso for Python versions below 3.11
  warnings.warn(


parsing log, completed traces ::   0%|          | 0/31509 [00:00<?, ?it/s]

In [4]:
df = pm4py.convert_to_dataframe(log)

In [5]:
df = df.loc[df["EventOrigin"] != "Workflow"].copy()

In [6]:
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 433444 entries, 0 to 1202265
Data columns (total 19 columns):
 #   Column                 Non-Null Count   Dtype              
---  ------                 --------------   -----              
 0   Action                 433444 non-null  object             
 1   org:resource           433444 non-null  object             
 2   concept:name           433444 non-null  object             
 3   EventOrigin            433444 non-null  object             
 4   EventID                433444 non-null  object             
 5   lifecycle:transition   433444 non-null  object             
 6   time:timestamp         433444 non-null  datetime64[ns, UTC]
 7   case:LoanGoal          433444 non-null  object             
 8   case:ApplicationType   433444 non-null  object             
 9   case:concept:name      433444 non-null  object             
 10  case:RequestedAmount   433444 non-null  float64            
 11  FirstWithdrawalAmount  42995 non-null   flo

In [7]:
df.isnull().any()

Action                   False
org:resource             False
concept:name             False
EventOrigin              False
EventID                  False
lifecycle:transition     False
time:timestamp           False
case:LoanGoal            False
case:ApplicationType     False
case:concept:name        False
case:RequestedAmount     False
FirstWithdrawalAmount     True
NumberOfTerms             True
Accepted                  True
MonthlyCost               True
Selected                  True
CreditScore               True
OfferedAmount             True
OfferID                   True
dtype: bool

In [8]:
df = df.drop(columns=['Action', 'EventOrigin', 'EventID', 'OfferID'])

In [9]:
df['case:concept:name'] = df['case:concept:name'].astype('string')
df['concept:name'] = df['concept:name'].astype('string')
df['lifecycle:transition'] = df['lifecycle:transition'].astype('string')

df['time:timestamp'] = pd.to_datetime(df['time:timestamp'], errors='coerce')

df['org:resource'] = df['org:resource'].astype('string')

df['case:LoanGoal'] = df['case:LoanGoal'].astype('string')
df['case:ApplicationType'] = df['case:ApplicationType'].astype('string')
df['Accepted'] = df['Accepted'].astype('string')
df['Selected'] = df['Selected'].astype('string')

df['case:RequestedAmount'] = df['case:RequestedAmount'].astype(np.float32)
df['FirstWithdrawalAmount'] = df['FirstWithdrawalAmount'].astype(np.float32)
df['NumberOfTerms'] = df['NumberOfTerms'].astype(np.float32)
df['MonthlyCost'] = df['MonthlyCost'].astype(np.float32)
df['CreditScore'] = df['CreditScore'].astype(np.float32)
df['OfferedAmount'] = df['OfferedAmount'].astype(np.float32)

In [10]:
df = df.sort_values(by=['case:concept:name', 'time:timestamp'], ascending=[True, True])

In [11]:
df['time_delta'] = df.groupby('case:concept:name')['time:timestamp'].diff()
df['time_delta'] = df['time_delta'].dt.total_seconds()
df['time_delta'] = df['time_delta'].fillna(0)

In [12]:
exclude_cols = ["case:concept:name", "time:timestamp"]

sorted_cols = sorted(
    [c for c in df.columns if c not in exclude_cols]
)

df = df[exclude_cols + sorted_cols]

In [13]:
df.head(20)

,case:concept:name,time:timestamp,Accepted,CreditScore,FirstWithdrawalAmount,MonthlyCost,NumberOfTerms,OfferedAmount,Selected,case:ApplicationType,case:LoanGoal,case:RequestedAmount,concept:name,lifecycle:transition,org:resource,time_delta
686058,Application_1000086665,2016-08-03 15:57:21.673000+00:00,<NA>,NaN,NaN,NaN,NaN,NaN,<NA>,New credit,"Other, see explanation",5000.0,A_Create Application,complete,User_1,0.000
686059,Application_1000086665,2016-08-03 15:57:21.734000+00:00,<NA>,NaN,NaN,NaN,NaN,NaN,<NA>,New credit,"Other, see explanation",5000.0,A_Submitted,complete,User_1,0.061
686063,Application_1000086665,2016-08-03 15:58:28.299000+00:00,<NA>,NaN,NaN,NaN,NaN,NaN,<NA>,New credit,"Other, see explanation",5000.0,A_Concept,complete,User_1,66.565
686066,Application_1000086665,2016-08-05 13:57:07.419000+00:00,<NA>,NaN,NaN,NaN,NaN,NaN,<NA>,New credit,"Other, see explanation",5000.0,A_Accepted,complete,User_5,165519.120
686067,Application_1000086665,2016-08-05 13:59:57.320000+00:00,True,0.0,5000.0,241.279999,22.0,5000.0,False,New credit,"Other, see explanation",5000.0,O_Create Offer,complete,User_5,169.901
686068,Application_1000086665,2016-08-05 13:59:58.162000+00:00,<NA>,NaN,NaN,NaN,NaN,NaN,<NA>,New credit,"Other, see explanation",5000.0,O_Created,complete,User_5,0.842
686069,Application_1000086665,2016-08-05 14:01:23.264000+00:00,<NA>,NaN,NaN,NaN,NaN,NaN,<NA>,New credit,"Other, see explanation",5000.0,O_Sent (mail and online),complete,User_5,85.102
686073,Application_1000086665,2016-08-05 14:01:23.288000+00:00,<NA>,NaN,NaN,NaN,NaN,NaN,<NA>,New credit,"Other, see explanation",5000.0,A_Complete,complete,User_5,0.024
686077,Application_1000086665,2016-09-05 06:00:36.710000+00:00,<NA>,NaN,NaN,NaN,NaN,NaN,<NA>,New credit,"Other, see explanation",5000.0,A_Cancelled,complete,User_1,2649553.422
686078,Application_1000086665,2016-09-05 06:00:36.829000+00:00,<NA>,NaN,NaN,NaN,NaN,NaN,<NA>,New credit,"Other, see explanation",5000.0,O_Cancelled,complete,User_1,0.119


In [14]:
num_cases = df['case:concept:name'].nunique()
print(f"Total number of unique cases: {num_cases}")

Total number of unique cases: 31509


### --- Feature Configurations ---

In [15]:
# --- Define feature specs ---
feature_specs = {
    
    "case:LoanGoal": {
        "type":            "categorical",
        "level":           "case",
        "vary":            True,
    },

    "case:ApplicationType": {
        "type":            "categorical",
        "level":           "case",
        "vary":            True,
    },

    "Accepted": {
        "type":           "categorical", 
        "level":          "event",
        "vary":           True
    },

    "Selected": {
        "type":           "categorical", 
        "level":          "event",
        "vary":           True
    },

    "org:resource": {
        "type":           "categorical",
        "level":          "event",
        "vary":           True,
    },

    "case:RequestedAmount": {
        "type":            "continuous",
        "level":           "case",
        "vary":            True,
        "quantile_low":    0.05,
        "quantile_high":   0.90, 
                 
    },
   
    "time_delta": {
        "type":            "continuous",
        "level":           "event",
        "vary":            True,
        "quantile_low":    0.20,
        "quantile_high":   0.80, 
    },

     "FirstWithdrawalAmount": {
        "type":            "continuous",
        "level":           "event",
        "vary":            True,
    },

     "NumberOfTerms": {
        "type":            "continuous",
        "level":           "event",
        "vary":            True,
    },

     "MonthlyCost": {
        "type":            "continuous",
        "level":           "event",
        "vary":            True,
    },

     "CreditScore": {
        "type":            "continuous",
        "level":           "event",
        "vary":            True,
        "quantile_low":    0.05,
        "quantile_high":   0.90, 
    },

    "OfferedAmount": {
        "type":            "continuous",
        "level":           "event",
        "vary":            True,
    },

    # immutable
    "concept:name": {
        "type":            "categorical", 
        "level":           "event",
        "vary":            False
    },

    "lifecycle:transition": {
        "type":           "categorical", 
        "level":          "event",
        "vary":           False
    },
}

In [16]:
feature_config = FeatureConfig.from_dataframe(
    df=df,
    feature_specs=feature_specs,
    activity_feature="concept:name",
    is_robust=True,
    default_quantile_low=0.05,
    default_quantile_high=0.95
)

feature_config.save()

C:\Users\dcoralage\Downloads\counterfactual_exp\config\feature_config.py:359: UserWarning: Feature 'FirstWithdrawalAmount' is heavily right-skewed (skew=2.32). Auto-tightening quantile_high to 0.95. Override by setting 'quantile_high' or 'permitted_range' in feature_specs.
  warnings.warn(
C:\Users\dcoralage\Downloads\counterfactual_exp\config\feature_config.py:359: UserWarning: Feature 'MonthlyCost' is heavily right-skewed (skew=3.47). Auto-tightening quantile_high to 0.95. Override by setting 'quantile_high' or 'permitted_range' in feature_specs.
  warnings.warn(


In [17]:
# feature_config = FeatureConfig.load()

In [18]:
feature_config.summary()

  activity_feature:    concept:name
  feature_order:       ['Accepted', 'CreditScore', 'FirstWithdrawalAmount', 'MonthlyCost', 'NumberOfTerms', 'OfferedAmount', 'Selected', 'case:ApplicationType', 'case:LoanGoal', 'case:RequestedAmount', 'concept:name', 'lifecycle:transition', 'org:resource', 'time_delta']
--------------------------------------------------------------------------------------------------------------------------------------
Feature                        Type           Level    Vary   Range/Categories                         MAD        Source              
--------------------------------------------------------------------------------------------------------------------------------------
case:LoanGoal                  categorical    case     yes    ['Boat', 'Business goal', 'Car', ...]    N/A        data_derived        
case:ApplicationType           categorical    case     yes    ['Limit raise', 'New credit']            N/A        data_derived        
Accepted         

### --- Next event prediction model ---

In [19]:
cat_cols = df.select_dtypes(include=["string"]).columns
for col in cat_cols:
    df[col] = df[col].fillna("NA").astype('string')

In [20]:
case_ids = df["case:concept:name"].unique()

train_cases, val_cases = train_test_split(case_ids, test_size=0.2, random_state=42)

train_df = df[df["case:concept:name"].isin(train_cases)].copy()
val_df   = df[df["case:concept:name"].isin(val_cases)].copy()

In [21]:
preprocessor_artifacts = PreprocessorArtifacts.build(
    df=train_df,
    feature_config=feature_config,      
    scaler_type="robust",
)

preprocessor_artifacts.save()

In [22]:
# preprocessor_artifacts = PreprocessorArtifacts.load()

In [23]:
preprocessor_artifacts.summary()

===================PreprocessorArtifacts====================
  scaler:              RobustScaler
  encoders:            ['case:LoanGoal', 'case:ApplicationType', 'Accepted', 'Selected', 'org:resource', 'concept:name', 'lifecycle:transition']
  activity_prototypes: 18 activities


In [24]:
# Transform nan cols to 0
float_cols = df.select_dtypes(include=["float32", "float64"]).columns
df[float_cols] = df[float_cols].fillna(0)
train_df[float_cols] = train_df[float_cols].fillna(0)
val_df[float_cols] = val_df[float_cols].fillna(0)

In [25]:
train_dataset = preprocessor_artifacts.transform_dataframe_to_processdataset(
    case_id_field="case:concept:name", 
    df=train_df,
    sort_field="time:timestamp"
)

val_dataset = preprocessor_artifacts.transform_dataframe_to_processdataset(
    case_id_field="case:concept:name", 
    df=val_df,
    sort_field="time:timestamp"
)

In [26]:
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False)

In [27]:
print(preprocessor_artifacts.get_categorical_feature_cardinality())

{'dynamic_categorical_info': {'Accepted': 3, 'Selected': 3, 'concept:name': 18, 'lifecycle:transition': 1, 'org:resource': 144}, 'static_categorical_info': {'case:ApplicationType': 2, 'case:LoanGoal': 14}}


In [28]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [29]:
device

device(type='cuda')

In [30]:
criterion = torch.nn.CrossEntropyLoss()

In [31]:
log_file, original_stdout = set_stdout_to_file(filepath="logs/bpic17-model_output.txt")

Epoch 020/100 | Train Loss: 0.1786 | LR: 9.05e-04
Epoch 040/100 | Train Loss: 0.1247 | LR: 6.55e-04
Epoch 060/100 | Train Loss: 0.0909 | LR: 3.46e-04
Epoch 080/100 | Train Loss: 0.0700 | LR: 9.64e-05
Epoch 100/100 | Train Loss: 0.0638 | LR: 1.00e-06
Time taken for next event model (training): 4076.320304 seconds
Time taken for next event model (validation): 2.108371 seconds
Val loss: {'loss': 0.8610628035990995, 'accuracy': 0.8946184159818997, 'f1_macro': 0.8424013816324873, 'f1_weighted': 0.8946177069284159}


In [32]:
embedding_metadata = preprocessor_artifacts.get_embedding_metadata()

model = ProcessLSTM(
    dynamic_categorical_info=embedding_metadata["dynamic_categorical_info"],
    static_categorical_info=embedding_metadata["static_categorical_info"],
    n_dynamic_continuous=embedding_metadata["n_dynamic_continuous"],
    n_static_continuous=embedding_metadata["n_static_continuous"],
    n_classes=embedding_metadata["n_classes"]
)

train_loss_history = train_ProcessLSTM(
    model=model,
    train_loader=train_loader,
    learning_rate=1e-3,
    criterion=criterion,
    num_epochs=100,
    device=device
)

model.save()

In [33]:
# model = ProcessLSTM.load()

In [34]:
val_loss = validate_ProcessLSTM(
    model=model,
    val_loader=val_loader,
    criterion=criterion,
    device=device
)

print("Val loss:", val_loss)

### --- Cleanup ---

In [35]:
if df["time:timestamp"].dt.tz is not None:
    df["time:timestamp"] = df["time:timestamp"].dt.tz_convert(None)
df.to_excel("../../data/bpic17.xlsx", index=False, engine="openpyxl")

In [36]:
sys.stdout = original_stdout
log_file.close()